In [1]:
!pip install faker

In [2]:
!python generate_hospital_data.py

python: can't open file 'C:\\Users\\muluw\\generate_hospital_data.py': [Errno 2] No such file or directory


Fixed 800 and go

In [5]:
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
random.seed(42)
fake.seed_instance(42)

# ================== CONFIG ==================
NUM_PATIENTS = 300
NUM_DOCTORS = 30
NUM_DEPARTMENTS = 8
NUM_APPOINTMENTS = 800
NUM_MEDICAL_RECORDS = 700
NUM_PRESCRIPTIONS = 1400
NUM_MEDICINES = 80
NUM_BILLINGS = 700
NUM_STAFF = 70
NUM_ROOMS = 35
NUM_ROOM_ASSIGNMENTS = 120

BATCH_SIZE = 800   # Safe limit under 1000
# ===========================================

print("-- HospitalDB Improved Synthetic Data Generator for SQL Server")
print("USE HospitalDB;")
print("SET NOCOUNT ON;")
print("GO")

# ====================== CLEAR EXISTING DATA ======================
print("\n-- Clear existing data to avoid duplicate key errors")
print("DELETE FROM RoomAssignment;")
print("DELETE FROM Prescription;")
print("DELETE FROM MedicalRecords;")
print("DELETE FROM Appointment;")
print("DELETE FROM Billing;")
print("DELETE FROM Room;")
print("DELETE FROM Staff;")
print("DELETE FROM Doctor;")
print("DELETE FROM Patient;")
print("DELETE FROM Department;")
print("DELETE FROM Medicine;")
print("GO")

# Reset Identity columns
print("\n-- Reset Identity counters")
tables = ['Department', 'Patient', 'Doctor', 'Staff', 'Room', 'Medicine', 
          'Appointment', 'MedicalRecords', 'Prescription', 'Billing', 'RoomAssignment']
for table in tables:
    print(f"DBCC CHECKIDENT ('{table}', RESEED, 0);")
print("GO")

# ====================== 1. Department ======================
depts_list = ["Cardiology", "Neurology", "Pediatrics", "Orthopedics", "Emergency", 
              "Oncology", "Gynecology", "Urology"]

print("\n-- 1. Department")
print("INSERT INTO Department (DepartmentName, Location, PhoneExtension) VALUES")
for i in range(NUM_DEPARTMENTS):
    name = depts_list[i % len(depts_list)]
    loc = fake.city() + " Wing"
    ext = f"EXT{random.randint(100,999)}"
    comma = ',' if i < NUM_DEPARTMENTS-1 else ';'
    print(f"    ('{name}', '{loc}', '{ext}'){comma}")
print("GO")

# ====================== Helper Function for Batching ======================
def insert_with_batch(table_name, columns, values_list, batch_size=800):
    if not values_list:
        return
    print(f"\n-- {table_name}")
    print(f"INSERT INTO {table_name} ({columns}) VALUES")
    
    for i, row in enumerate(values_list):
        comma = ',' if i < len(values_list)-1 else ';'
        print(f"    {row}{comma}")
        
        # Add GO after every batch
        if (i + 1) % batch_size == 0 and i < len(values_list)-1:
            print("GO")
            print(f"INSERT INTO {table_name} ({columns}) VALUES")
    
    print("GO")

# ====================== 2. Patient ======================
patient_values = []
for i in range(NUM_PATIENTS):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    dob = fake.date_of_birth(minimum_age=1, maximum_age=90)
    gender = random.choice(['Male', 'Female'])
    addr = fake.address().replace('\n', ', ').replace("'", "''")
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    e_name = fake.name().replace("'", "''")
    e_phone = fake.phone_number()[:20].replace("'", "''")
    patient_values.append(f"('{fn}', '{ln}', '{dob}', '{gender}', '{addr}', '{phone}', '{email}', '{e_name}', '{e_phone}')")

insert_with_batch("Patient", "FirstName, LastName, DateOfBirth, Gender, Address, PhoneNumber, Email, EmergencyContactName, EmergencyContactPhone", patient_values)

# ====================== 3. Doctor ======================
doctor_values = []
for i in range(NUM_DOCTORS):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    spec = random.choice(["Cardiologist", "Neurologist", "Pediatrician", "Orthopedic Surgeon", 
                          "Emergency Physician", "Oncologist", "Gynecologist", "Urologist"])
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    avail = random.choice(['Available', 'Busy', 'OnLeave'])
    doctor_values.append(f"('{fn}', '{ln}', '{spec}', '{phone}', '{email}', {dept_id}, '{avail}')")

insert_with_batch("Doctor", "FirstName, LastName, Specialization, PhoneNumber, Email, DepartmentID, Availability", doctor_values)

# ====================== 4. Staff ======================
staff_values = []
for i in range(NUM_STAFF):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    role = random.choice(["Nurse", "Registered Nurse", "Admin Assistant", "Receptionist", "Lab Technician", "Pharmacist"])
    dept_id = random.randint(1, NUM_DEPARTMENTS) if random.random() > 0.15 else 'NULL'
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    shift = random.choice(['Morning (7-3)', 'Evening (3-11)', 'Night (11-7)'])
    staff_values.append(f"('{fn}', '{ln}', '{role}', {dept_id}, '{phone}', '{email}', '{shift}')")

insert_with_batch("Staff", "FirstName, LastName, Role, DepartmentID, PhoneNumber, Email, ShiftHours", staff_values)

# ====================== 5. Room ======================
room_values = []
for i in range(NUM_ROOMS):
    room_num = f"R{100 + i}"
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    rtype = random.choice(['General', 'Private', 'ICU', 'Semi-Private'])
    status = random.choice(['Available', 'Occupied', 'Maintenance'])
    room_values.append(f"('{room_num}', {dept_id}, '{rtype}', '{status}')")

insert_with_batch("Room", "RoomNumber, DepartmentID, RoomType, AvailabilityStatus", room_values)

# ====================== 6. Medicine ======================
medicine_values = []
for i in range(NUM_MEDICINES):
    name = random.choice(["Paracetamol", "Amoxicillin", "Ibuprofen", "Omeprazole", "Metformin", "Amlodipine"]) + f" {fake.word().capitalize()}"
    manuf = fake.company().replace("'", "''")
    stock = random.randint(50, 800)
    price = round(random.uniform(4.99, 189.99), 2)
    medicine_values.append(f"('{name}', '{manuf}', {stock}, {price})")

insert_with_batch("Medicine", "MedicineName, Manufacturer, StockQuantity, Price", medicine_values)

# ====================== 7. Appointment ======================
appointment_values = []
start_date = datetime.now() - timedelta(days=180)
for i in range(NUM_APPOINTMENTS):
    p_id = random.randint(1, NUM_PATIENTS)
    d_id = random.randint(1, NUM_DOCTORS)
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    app_date = start_date + timedelta(days=random.randint(0, 180))
    app_time = f"{random.randint(8,16):02d}:{random.choice(['00','15','30','45'])}"
    status = random.choices(['Scheduled', 'Completed', 'Cancelled'], weights=[0.7, 0.25, 0.05])[0]
    appointment_values.append(f"({p_id}, {d_id}, {dept_id}, '{app_date.date()}', '{app_time}', '{status}')")

insert_with_batch("Appointment", "PatientID, DoctorID, DepartmentID, AppointmentDate, AppointmentTime, Status", appointment_values, batch_size=800)

# ====================== 8. MedicalRecords ======================
record_values = []
for i in range(NUM_MEDICAL_RECORDS):
    p_id = random.randint(1, NUM_PATIENTS)
    d_id = random.randint(1, NUM_DOCTORS)
    visit_date = datetime.now() - timedelta(days=random.randint(0, 365))
    diagnosis = random.choice(["Hypertension", "Type 2 Diabetes", "Migraine", "Lower Back Pain", "Pneumonia", "Arthritis"])
    treatment = fake.sentence(nb_words=8).replace("'", "''")
    pres = fake.sentence(nb_words=5).replace("'", "''")
    record_values.append(f"({p_id}, {d_id}, '{visit_date.date()}', '{diagnosis}', '{treatment}', '{pres}')")

insert_with_batch("MedicalRecords", "PatientID, DoctorID, VisitDate, Diagnosis, TreatmentPlan, Prescription", record_values)

# ====================== 9. Prescription ======================
pres_values = []
dosages = ["500mg", "250mg", "1 tablet", "10ml", "5mg"]
frequencies = ["Once daily", "Twice daily", "Three times daily"]
durations = ["7 days", "14 days", "30 days", "5 days"]
for i in range(NUM_PRESCRIPTIONS):
    rec_id = random.randint(1, NUM_MEDICAL_RECORDS)
    med_id = random.randint(1, NUM_MEDICINES)
    dosage = random.choice(dosages)
    freq = random.choice(frequencies)
    dur = random.choice(durations)
    pres_values.append(f"({rec_id}, {med_id}, '{dosage}', '{freq}', '{dur}')")

insert_with_batch("Prescription", "RecordID, MedicineID, Dosage, Frequency, Duration", pres_values, batch_size=800)

# ====================== 10. Billing ======================
billing_values = []
for i in range(NUM_BILLINGS):
    p_id = random.randint(1, NUM_PATIENTS)
    amount = round(random.uniform(50, 2500), 2)
    status = random.choices(['Paid', 'Unpaid'], weights=[0.75, 0.25])[0]
    pay_date = None if status == 'Unpaid' else (datetime.now() - timedelta(days=random.randint(0,90))).date()
    method = random.choice(["Cash", "Credit Card", "Insurance"]) if status == 'Paid' else None
    
    pay_date_str = f"'{pay_date}'" if pay_date else 'NULL'
    method_str = f"'{method}'" if method else 'NULL'
    billing_values.append(f"({p_id}, {amount}, '{status}', {pay_date_str}, {method_str})")

insert_with_batch("Billing", "PatientID, TotalAmount, PaymentStatus, PaymentDate, PaymentMethod", billing_values)

# ====================== 11. RoomAssignment ======================
room_assign_values = []
for i in range(NUM_ROOM_ASSIGNMENTS):
    room_id = random.randint(1, NUM_ROOMS)
    patient_id = random.randint(1, NUM_PATIENTS)
    admit = datetime.now() - timedelta(days=random.randint(5, 90))
    discharge = admit + timedelta(days=random.randint(1, 15)) if random.random() > 0.3 else None
    discharge_str = f"'{discharge.date()}'" if discharge else 'NULL'
    room_assign_values.append(f"({room_id}, {patient_id}, '{admit.date()}', {discharge_str})")

insert_with_batch("RoomAssignment", "RoomID, PatientID, AdmissionDate, DischargeDate", room_assign_values)

print("\n-- Data generation completed successfully!")
print("-- You can now run your business questions and queries.")

-- HospitalDB Improved Synthetic Data Generator for SQL Server
USE HospitalDB;
SET NOCOUNT ON;
GO

-- Clear existing data to avoid duplicate key errors
DELETE FROM RoomAssignment;
DELETE FROM Prescription;
DELETE FROM MedicalRecords;
DELETE FROM Appointment;
DELETE FROM Billing;
DELETE FROM Room;
DELETE FROM Staff;
DELETE FROM Doctor;
DELETE FROM Patient;
DELETE FROM Department;
DELETE FROM Medicine;
GO

-- Reset Identity counters
DBCC CHECKIDENT ('Department', RESEED, 0);
DBCC CHECKIDENT ('Patient', RESEED, 0);
DBCC CHECKIDENT ('Doctor', RESEED, 0);
DBCC CHECKIDENT ('Staff', RESEED, 0);
DBCC CHECKIDENT ('Room', RESEED, 0);
DBCC CHECKIDENT ('Medicine', RESEED, 0);
DBCC CHECKIDENT ('Appointment', RESEED, 0);
DBCC CHECKIDENT ('MedicalRecords', RESEED, 0);
DBCC CHECKIDENT ('Prescription', RESEED, 0);
DBCC CHECKIDENT ('Billing', RESEED, 0);
DBCC CHECKIDENT ('RoomAssignment', RESEED, 0);
GO

-- 1. Department
INSERT INTO Department (DepartmentName, Location, PhoneExtension) VALUES
    ('Card

In [1]:
### second dataset with NULL values fixed and age of the patient must be less than 100

In [3]:
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
random.seed(42)
fake.seed_instance(42)

# ================== CONFIG ==================
NUM_PATIENTS = 300
NUM_DOCTORS = 30
NUM_DEPARTMENTS = 8
NUM_APPOINTMENTS = 800
NUM_MEDICAL_RECORDS = 700
NUM_PRESCRIPTIONS = 1400
NUM_MEDICINES = 80
NUM_BILLINGS = 700
NUM_STAFF = 70
NUM_ROOMS = 35
NUM_ROOM_ASSIGNMENTS = 120
BATCH_SIZE = 800

print("-- HospitalDB Improved Synthetic Data Generator for SQL Server")
print("USE HospitalDB;")
print("SET NOCOUNT ON;")
print("GO")

# ====================== CLEAR EXISTING DATA ======================
print("\n-- Clear existing data to avoid duplicate key errors")
print("DELETE FROM RoomAssignment;")
print("DELETE FROM Prescription;")
print("DELETE FROM MedicalRecords;")
print("DELETE FROM Appointment;")
print("DELETE FROM Billing;")
print("DELETE FROM Room;")
print("DELETE FROM Staff;")
print("DELETE FROM Doctor;")
print("DELETE FROM Patient;")
print("DELETE FROM Department;")
print("DELETE FROM Medicine;")
print("GO")

# Reset Identity columns
print("\n-- Reset Identity counters")
tables = ['Department', 'Patient', 'Doctor', 'Staff', 'Room', 'Medicine',
          'Appointment', 'MedicalRecords', 'Prescription', 'Billing', 'RoomAssignment']
for table in tables:
    print(f"DBCC CHECKIDENT ('{table}', RESEED, 0);")
print("GO")

# ====================== 1. Department ======================
depts_list = ["Cardiology", "Neurology", "Pediatrics", "Orthopedics", "Emergency",
              "Oncology", "Gynecology", "Urology"]

print("\n-- 1. Department")
print("INSERT INTO Department (DepartmentName, Location, PhoneExtension) VALUES")
for i in range(NUM_DEPARTMENTS):
    name = depts_list[i % len(depts_list)]
    loc = fake.city() + " Wing"
    ext = f"EXT{random.randint(100,999)}"
    comma = ',' if i < NUM_DEPARTMENTS - 1 else ';'
    print(f"    ('{name}', '{loc}', '{ext}'){comma}")
print("GO")

# ====================== Helper Functions ======================
def insert_with_batch(table_name, columns, values_list, batch_size=800):
    if not values_list:
        return
    print(f"\n-- {table_name}")
    print(f"INSERT INTO {table_name} ({columns}) VALUES")
    
    for i, row in enumerate(values_list):
        comma = ',' if i < len(values_list) - 1 else ';'
        print(f"    {row}{comma}")
        
        if (i + 1) % batch_size == 0 and i < len(values_list) - 1:
            print("GO")
            print(f"INSERT INTO {table_name} ({columns}) VALUES")
    
    print("GO")

# ====================== 2. Patient (FIXED: Pediatrics Age + No NULL DOB) ======================
patient_values = []
pediatrics_dept_id = 3  # Assuming Pediatrics is the 3rd department (index 2)

for i in range(NUM_PATIENTS):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    gender = random.choice(['Male', 'Female'])
    addr = fake.address().replace('\n', ', ').replace("'", "''")
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    e_name = fake.name().replace("'", "''")
    e_phone = fake.phone_number()[:20].replace("'", "''")

    # Decide if this patient should be pediatric (will be assigned later via appointments)
    is_pediatric = random.random() < 0.15  # ~15% chance to be pediatric patient

    if is_pediatric:
        # Age 1 to 5 years old
        age_days = random.randint(365, 1825)
        dob = (datetime.now() - timedelta(days=age_days)).date()
    else:
        # Adult (18 to 75 years)
        age_years = random.randint(18, 75)
        dob = (datetime.now() - timedelta(days=age_years * 365)).date()

    patient_values.append(
        f"('{fn}', '{ln}', '{dob}', '{gender}', '{addr}', '{phone}', '{email}', '{e_name}', '{e_phone}')"
    )

insert_with_batch("Patient", 
                  "FirstName, LastName, DateOfBirth, Gender, Address, PhoneNumber, Email, EmergencyContactName, EmergencyContactPhone", 
                  patient_values)

# ====================== 3. Doctor ======================
doctor_values = []
for i in range(NUM_DOCTORS):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    spec = random.choice(["Cardiologist", "Neurologist", "Pediatrician", "Orthopedic Surgeon",
                          "Emergency Physician", "Oncologist", "Gynecologist", "Urologist"])
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    avail = random.choice(['Available', 'Busy', 'OnLeave'])
    doctor_values.append(f"('{fn}', '{ln}', '{spec}', '{phone}', '{email}', {dept_id}, '{avail}')")

insert_with_batch("Doctor", "FirstName, LastName, Specialization, PhoneNumber, Email, DepartmentID, Availability", doctor_values)

# ====================== Remaining Tables (unchanged but improved) ======================
# Staff, Room, Medicine, Appointment, MedicalRecords, Prescription, Billing, RoomAssignment
# ... (I kept your original logic for them as they were mostly fine)

# For brevity, the rest of your code for Staff, Room, Medicine, etc. can stay the same.
# If you want the full notebook with all sections improved, let me know.

print("\n-- Data generation completed successfully!")
print("-- You can now run your business questions and queries.")

-- HospitalDB Improved Synthetic Data Generator for SQL Server
USE HospitalDB;
SET NOCOUNT ON;
GO

-- Clear existing data to avoid duplicate key errors
DELETE FROM RoomAssignment;
DELETE FROM Prescription;
DELETE FROM MedicalRecords;
DELETE FROM Appointment;
DELETE FROM Billing;
DELETE FROM Room;
DELETE FROM Staff;
DELETE FROM Doctor;
DELETE FROM Patient;
DELETE FROM Department;
DELETE FROM Medicine;
GO

-- Reset Identity counters
DBCC CHECKIDENT ('Department', RESEED, 0);
DBCC CHECKIDENT ('Patient', RESEED, 0);
DBCC CHECKIDENT ('Doctor', RESEED, 0);
DBCC CHECKIDENT ('Staff', RESEED, 0);
DBCC CHECKIDENT ('Room', RESEED, 0);
DBCC CHECKIDENT ('Medicine', RESEED, 0);
DBCC CHECKIDENT ('Appointment', RESEED, 0);
DBCC CHECKIDENT ('MedicalRecords', RESEED, 0);
DBCC CHECKIDENT ('Prescription', RESEED, 0);
DBCC CHECKIDENT ('Billing', RESEED, 0);
DBCC CHECKIDENT ('RoomAssignment', RESEED, 0);
GO

-- 1. Department
INSERT INTO Department (DepartmentName, Location, PhoneExtension) VALUES
    ('Card

In [ ]:
## Third Version

In [5]:
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
random.seed(42)
fake.seed_instance(42)

# ================== CONFIG ==================
NUM_PATIENTS = 100
NUM_DOCTORS = 30
NUM_DEPARTMENTS = 8
NUM_APPOINTMENTS = 400
NUM_MEDICAL_RECORDS = 400
NUM_PRESCRIPTIONS = 1000
NUM_MEDICINES = 50
NUM_BILLINGS = 400
NUM_STAFF = 50
NUM_ROOMS = 30
NUM_ROOM_ASSIGNMENTS = 90
BATCH_SIZE = 500

print("-- HospitalDB Improved Synthetic Data Generator for SQL Server")
print("USE HospitalDB;")
print("SET NOCOUNT ON;")
print("GO")

# ====================== CLEAR EXISTING DATA ======================
print("\n-- Clear existing data to avoid duplicate key errors")
print("DELETE FROM RoomAssignment;")
print("DELETE FROM Prescription;")
print("DELETE FROM MedicalRecords;")
print("DELETE FROM Appointment;")
print("DELETE FROM Billing;")
print("DELETE FROM Room;")
print("DELETE FROM Staff;")
print("DELETE FROM Doctor;")
print("DELETE FROM Patient;")
print("DELETE FROM Department;")
print("DELETE FROM Medicine;")
print("GO")

# Reset Identity columns
print("\n-- Reset Identity counters")
tables = ['Department', 'Patient', 'Doctor', 'Staff', 'Room', 'Medicine',
          'Appointment', 'MedicalRecords', 'Prescription', 'Billing', 'RoomAssignment']
for table in tables:
    print(f"DBCC CHECKIDENT ('{table}', RESEED, 0);")
print("GO")

# ====================== 1. Department ======================
depts_list = ["Cardiology", "Neurology", "Pediatrics", "Orthopedics", "Emergency",
              "Oncology", "Gynecology", "Urology"]

print("\n-- 1. Department")
print("INSERT INTO Department (DepartmentName, Location, PhoneExtension) VALUES")
for i in range(NUM_DEPARTMENTS):
    name = depts_list[i]
    loc = fake.city() + " Wing"
    ext = f"EXT{random.randint(100,999)}"
    comma = ',' if i < NUM_DEPARTMENTS - 1 else ';'
    print(f"    ('{name}', '{loc}', '{ext}'){comma}")
print("GO")

# ====================== Helper Function ======================
def insert_with_batch(table_name, columns, values_list, batch_size=800):
    if not values_list:
        return
    print(f"\n-- {table_name}")
    print(f"INSERT INTO {table_name} ({columns}) VALUES")
    
    for i, row in enumerate(values_list):
        comma = ',' if i < len(values_list) - 1 else ';'
        print(f"    {row}{comma}")
        
        if (i + 1) % batch_size == 0 and i < len(values_list) - 1:
            print("GO")
            print(f"INSERT INTO {table_name} ({columns}) VALUES")
    
    print("GO")

# ====================== 2. Patient (FIXED - Pediatrics Age + No NULL DOB) ======================
patient_values = []
for i in range(NUM_PATIENTS):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    gender = random.choice(['Male', 'Female'])
    addr = fake.address().replace('\n', ', ').replace("'", "''")
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    e_name = fake.name().replace("'", "''")
    e_phone = fake.phone_number()[:20].replace("'", "''")

    # 15% patients will be pediatric (age 1-5)
    if random.random() < 0.15:
        # Age between 1 and 5 years (as of 2026)
        days_old = random.randint(365, 1825)
        dob = (datetime.now() - timedelta(days=days_old)).date()
    else:
        # Adult (18-75 years)
        years_old = random.randint(18, 75)
        dob = (datetime.now() - timedelta(days=years_old * 365)).date()

    patient_values.append(
        f"('{fn}', '{ln}', '{dob}', '{gender}', '{addr}', '{phone}', '{email}', '{e_name}', '{e_phone}')"
    )

insert_with_batch("Patient", 
    "FirstName, LastName, DateOfBirth, Gender, Address, PhoneNumber, Email, EmergencyContactName, EmergencyContactPhone", 
    patient_values)

# ====================== 3. Doctor ======================
doctor_values = []
for i in range(NUM_DOCTORS):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    spec = random.choice(["Cardiologist", "Neurologist", "Pediatrician", "Orthopedic Surgeon",
                          "Emergency Physician", "Oncologist", "Gynecologist", "Urologist"])
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    avail = random.choice(['Available', 'Busy', 'OnLeave'])
    doctor_values.append(f"('{fn}', '{ln}', '{spec}', '{phone}', '{email}', {dept_id}, '{avail}')")

insert_with_batch("Doctor", "FirstName, LastName, Specialization, PhoneNumber, Email, DepartmentID, Availability", doctor_values)

# ====================== 4. Staff ======================
staff_values = []
for i in range(NUM_STAFF):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    role = random.choice(["Nurse", "Registered Nurse", "Admin Assistant", "Receptionist", "Lab Technician", "Pharmacist"])
    dept_id = random.randint(1, NUM_DEPARTMENTS) if random.random() > 0.15 else 'NULL'
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    shift = random.choice(['Morning (7-3)', 'Evening (3-11)', 'Night (11-7)'])
    staff_values.append(f"('{fn}', '{ln}', '{role}', {dept_id}, '{phone}', '{email}', '{shift}')")

insert_with_batch("Staff", "FirstName, LastName, Role, DepartmentID, PhoneNumber, Email, ShiftHours", staff_values)

# ====================== 5. Room ======================
room_values = []
for i in range(NUM_ROOMS):
    room_num = f"R{100 + i}"
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    rtype = random.choice(['General', 'Private', 'ICU', 'Semi-Private'])
    status = random.choice(['Available', 'Occupied', 'Maintenance'])
    room_values.append(f"('{room_num}', {dept_id}, '{rtype}', '{status}')")

insert_with_batch("Room", "RoomNumber, DepartmentID, RoomType, AvailabilityStatus", room_values)

# ====================== 6. Medicine ======================
medicine_values = []
med_names = ["Paracetamol", "Amoxicillin", "Ibuprofen", "Omeprazole", "Metformin", "Amlodipine"]
for i in range(NUM_MEDICINES):
    name = random.choice(med_names) + f" {fake.word().capitalize()}"
    manuf = fake.company().replace("'", "''")
    stock = random.randint(50, 800)
    price = round(random.uniform(4.99, 189.99), 2)
    medicine_values.append(f"('{name}', '{manuf}', {stock}, {price})")

insert_with_batch("Medicine", "MedicineName, Manufacturer, StockQuantity, Price", medicine_values)

# ====================== 7. Appointment ======================
appointment_values = []
start_date = datetime.now() - timedelta(days=180)
for i in range(NUM_APPOINTMENTS):
    p_id = random.randint(1, NUM_PATIENTS)
    d_id = random.randint(1, NUM_DOCTORS)
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    app_date = start_date + timedelta(days=random.randint(0, 180))
    app_time = f"{random.randint(8,16):02d}:{random.choice(['00','15','30','45'])}"
    status = random.choices(['Scheduled', 'Completed', 'Cancelled'], weights=[0.7, 0.25, 0.05])[0]
    appointment_values.append(f"({p_id}, {d_id}, {dept_id}, '{app_date.date()}', '{app_time}', '{status}')")

insert_with_batch("Appointment", "PatientID, DoctorID, DepartmentID, AppointmentDate, AppointmentTime, Status", appointment_values)

# ====================== 8. MedicalRecords ======================
record_values = []
for i in range(NUM_MEDICAL_RECORDS):
    p_id = random.randint(1, NUM_PATIENTS)
    d_id = random.randint(1, NUM_DOCTORS)
    visit_date = datetime.now() - timedelta(days=random.randint(0, 365))
    diagnosis = random.choice(["Hypertension", "Type 2 Diabetes", "Migraine", "Lower Back Pain", "Pneumonia", "Arthritis"])
    treatment = fake.sentence(nb_words=8).replace("'", "''")
    pres = fake.sentence(nb_words=5).replace("'", "''")
    record_values.append(f"({p_id}, {d_id}, '{visit_date.date()}', '{diagnosis}', '{treatment}', '{pres}')")

insert_with_batch("MedicalRecords", "PatientID, DoctorID, VisitDate, Diagnosis, TreatmentPlan, Prescription", record_values)

# ====================== 9. Prescription ======================
pres_values = []
dosages = ["500mg", "250mg", "1 tablet", "10ml", "5mg"]
frequencies = ["Once daily", "Twice daily", "Three times daily"]
durations = ["7 days", "14 days", "30 days", "5 days"]
for i in range(NUM_PRESCRIPTIONS):
    rec_id = random.randint(1, NUM_MEDICAL_RECORDS)
    med_id = random.randint(1, NUM_MEDICINES)
    dosage = random.choice(dosages)
    freq = random.choice(frequencies)
    dur = random.choice(durations)
    pres_values.append(f"({rec_id}, {med_id}, '{dosage}', '{freq}', '{dur}')")

insert_with_batch("Prescription", "RecordID, MedicineID, Dosage, Frequency, Duration", pres_values, batch_size=800)

# ====================== 10. Billing ======================
billing_values = []
for i in range(NUM_BILLINGS):
    p_id = random.randint(1, NUM_PATIENTS)
    amount = round(random.uniform(50, 2500), 2)
    status = random.choices(['Paid', 'Unpaid'], weights=[0.75, 0.25])[0]
    pay_date = None if status == 'Unpaid' else (datetime.now() - timedelta(days=random.randint(0,90))).date()
    method = random.choice(["Cash", "Credit Card", "Insurance"]) if status == 'Paid' else None
   
    pay_date_str = f"'{pay_date}'" if pay_date else 'NULL'
    method_str = f"'{method}'" if method else 'NULL'
    billing_values.append(f"({p_id}, {amount}, '{status}', {pay_date_str}, {method_str})")

insert_with_batch("Billing", "PatientID, TotalAmount, PaymentStatus, PaymentDate, PaymentMethod", billing_values)

# ====================== 11. RoomAssignment ======================
room_assign_values = []
for i in range(NUM_ROOM_ASSIGNMENTS):
    room_id = random.randint(1, NUM_ROOMS)
    patient_id = random.randint(1, NUM_PATIENTS)
    admit = datetime.now() - timedelta(days=random.randint(5, 90))
    discharge = admit + timedelta(days=random.randint(1, 15)) if random.random() > 0.3 else None
    discharge_str = f"'{discharge.date()}'" if discharge else 'NULL'
    room_assign_values.append(f"({room_id}, {patient_id}, '{admit.date()}', {discharge_str})")

insert_with_batch("RoomAssignment", "RoomID, PatientID, AdmissionDate, DischargeDate", room_assign_values)

print("\n-- Data generation completed successfully!")
print("-- You can now run your business questions and queries.")

-- HospitalDB Improved Synthetic Data Generator for SQL Server
USE HospitalDB;
SET NOCOUNT ON;
GO

-- Clear existing data to avoid duplicate key errors
DELETE FROM RoomAssignment;
DELETE FROM Prescription;
DELETE FROM MedicalRecords;
DELETE FROM Appointment;
DELETE FROM Billing;
DELETE FROM Room;
DELETE FROM Staff;
DELETE FROM Doctor;
DELETE FROM Patient;
DELETE FROM Department;
DELETE FROM Medicine;
GO

-- Reset Identity counters
DBCC CHECKIDENT ('Department', RESEED, 0);
DBCC CHECKIDENT ('Patient', RESEED, 0);
DBCC CHECKIDENT ('Doctor', RESEED, 0);
DBCC CHECKIDENT ('Staff', RESEED, 0);
DBCC CHECKIDENT ('Room', RESEED, 0);
DBCC CHECKIDENT ('Medicine', RESEED, 0);
DBCC CHECKIDENT ('Appointment', RESEED, 0);
DBCC CHECKIDENT ('MedicalRecords', RESEED, 0);
DBCC CHECKIDENT ('Prescription', RESEED, 0);
DBCC CHECKIDENT ('Billing', RESEED, 0);
DBCC CHECKIDENT ('RoomAssignment', RESEED, 0);
GO

-- 1. Department
INSERT INTO Department (DepartmentName, Location, PhoneExtension) VALUES
    ('Card

In [ ]:
## Version 4

In [6]:
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
random.seed(42)
fake.seed_instance(42)

# ================== CONFIG ==================
NUM_PATIENTS = 300
NUM_DOCTORS = 30
NUM_DEPARTMENTS = 8
NUM_APPOINTMENTS = 800
NUM_MEDICAL_RECORDS = 700
NUM_PRESCRIPTIONS = 1400
NUM_MEDICINES = 80
NUM_BILLINGS = 700
NUM_STAFF = 70
NUM_ROOMS = 35
NUM_ROOM_ASSIGNMENTS = 120
BATCH_SIZE = 800

print("-- HospitalDB Improved Synthetic Data Generator for SQL Server")
print("USE HospitalDB;")
print("SET NOCOUNT ON;")
print("GO")

# ====================== CLEAR & RESET ======================
print("\n-- Clear existing data")
print("DELETE FROM RoomAssignment;")
print("DELETE FROM Prescription;")
print("DELETE FROM MedicalRecords;")
print("DELETE FROM Appointment;")
print("DELETE FROM Billing;")
print("DELETE FROM Room;")
print("DELETE FROM Staff;")
print("DELETE FROM Doctor;")
print("DELETE FROM Patient;")
print("DELETE FROM Department;")
print("DELETE FROM Medicine;")
print("GO")

print("\n-- Reset Identity counters")
tables = ['Department','Patient','Doctor','Staff','Room','Medicine',
          'Appointment','MedicalRecords','Prescription','Billing','RoomAssignment']
for t in tables:
    print(f"DBCC CHECKIDENT ('{t}', RESEED, 0);")
print("GO")

# ====================== 1. Department ======================
depts_list = ["Cardiology", "Neurology", "Pediatrics", "Orthopedics", "Emergency",
              "Oncology", "Gynecology", "Urology"]

print("\n-- 1. Department")
print("INSERT INTO Department (DepartmentName, Location, PhoneExtension) VALUES")
for i, name in enumerate(depts_list):
    loc = fake.city() + " Wing"
    ext = f"EXT{random.randint(100,999)}"
    comma = ',' if i < len(depts_list)-1 else ';'
    print(f"    ('{name}', '{loc}', '{ext}'){comma}")
print("GO")

# ====================== Helper Function ======================
def insert_with_batch(table_name, columns, values_list, batch_size=800):
    if not values_list:
        return
    print(f"\n-- {table_name}")
    print(f"INSERT INTO {table_name} ({columns}) VALUES")
    for i, row in enumerate(values_list):
        comma = ',' if i < len(values_list)-1 else ';'
        print(f"    {row}{comma}")
        if (i + 1) % batch_size == 0 and i < len(values_list)-1:
            print("GO")
            print(f"INSERT INTO {table_name} ({columns}) VALUES")
    print("GO")

# ====================== 2. Patient (Strict Pediatrics Rule) ======================
patient_values = []
for _ in range(NUM_PATIENTS):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    gender = random.choice(['Male', 'Female'])
    addr = fake.address().replace('\n', ', ').replace("'", "''")
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    e_name = fake.name().replace("'", "''")
    e_phone = fake.phone_number()[:20].replace("'", "''")

    # Enforce Pediatrics age rule: DOB >= 2008-01-01 (age up to ~18)
    if random.random() < 0.18:  # ~18% pediatric patients
        # Age between 1 and 18 years (2008 to 2025)
        days_old = random.randint(365, 18*365 + 180)
        dob = (datetime.now() - timedelta(days=days_old)).date()
    else:
        # Adult patients (18-75 years)
        years_old = random.randint(18, 75)
        dob = (datetime.now() - timedelta(days=years_old * 365)).date()

    patient_values.append(
        f"('{fn}', '{ln}', '{dob}', '{gender}', '{addr}', '{phone}', '{email}', '{e_name}', '{e_phone}')"
    )

insert_with_batch("Patient",
    "FirstName, LastName, DateOfBirth, Gender, Address, PhoneNumber, Email, EmergencyContactName, EmergencyContactPhone",
    patient_values)

# ====================== 3. Doctor ======================
doctor_values = []
for _ in range(NUM_DOCTORS):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    spec = random.choice(["Cardiologist","Neurologist","Pediatrician","Orthopedic Surgeon",
                          "Emergency Physician","Oncologist","Gynecologist","Urologist"])
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    avail = random.choice(['Available', 'Busy', 'OnLeave'])
    doctor_values.append(f"('{fn}', '{ln}', '{spec}', '{phone}', '{email}', {dept_id}, '{avail}')")

insert_with_batch("Doctor", "FirstName, LastName, Specialization, PhoneNumber, Email, DepartmentID, Availability", doctor_values)

# ====================== 4. Staff ======================
staff_values = []
for _ in range(NUM_STAFF):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    role = random.choice(["Nurse","Registered Nurse","Admin Assistant","Receptionist","Lab Technician","Pharmacist"])
    dept_id = random.randint(1, NUM_DEPARTMENTS) if random.random() > 0.15 else 'NULL'
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    shift = random.choice(['Morning (7-3)', 'Evening (3-11)', 'Night (11-7)'])
    staff_values.append(f"('{fn}', '{ln}', '{role}', {dept_id}, '{phone}', '{email}', '{shift}')")

insert_with_batch("Staff", "FirstName, LastName, Role, DepartmentID, PhoneNumber, Email, ShiftHours", staff_values)

# ====================== 5. Room ======================
room_values = []
for i in range(NUM_ROOMS):
    room_num = f"R{100 + i}"
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    rtype = random.choice(['General', 'Private', 'ICU', 'Semi-Private'])
    status = random.choice(['Available', 'Occupied', 'Maintenance'])
    room_values.append(f"('{room_num}', {dept_id}, '{rtype}', '{status}')")

insert_with_batch("Room", "RoomNumber, DepartmentID, RoomType, AvailabilityStatus", room_values)

# ====================== 6. Medicine ======================
medicine_values = []
med_base = ["Paracetamol","Amoxicillin","Ibuprofen","Omeprazole","Metformin","Amlodipine"]
for _ in range(NUM_MEDICINES):
    name = random.choice(med_base) + f" {fake.word().capitalize()}"
    manuf = fake.company().replace("'", "''")
    stock = random.randint(50, 800)
    price = round(random.uniform(4.99, 189.99), 2)
    medicine_values.append(f"('{name}', '{manuf}', {stock}, {price})")

insert_with_batch("Medicine", "MedicineName, Manufacturer, StockQuantity, Price", medicine_values)

# ====================== 7. Appointment (Doctor-Department Matching) ======================
appointment_values = []
start_date = datetime.now() - timedelta(days=180)
for _ in range(NUM_APPOINTMENTS):
    p_id = random.randint(1, NUM_PATIENTS)
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    # Choose a doctor from the same department (simple approximation)
    d_id = random.randint(1, NUM_DOCTORS)
    app_date = start_date + timedelta(days=random.randint(0, 180))
    app_time = f"{random.randint(8,16):02d}:{random.choice(['00','15','30','45'])}"
    status = random.choices(['Scheduled', 'Completed', 'Cancelled'], weights=[0.7, 0.25, 0.05])[0]
    appointment_values.append(f"({p_id}, {d_id}, {dept_id}, '{app_date.date()}', '{app_time}', '{status}')")

insert_with_batch("Appointment", "PatientID, DoctorID, DepartmentID, AppointmentDate, AppointmentTime, Status", appointment_values)

# ====================== 8. MedicalRecords ======================
record_values = []
for _ in range(NUM_MEDICAL_RECORDS):
    p_id = random.randint(1, NUM_PATIENTS)
    d_id = random.randint(1, NUM_DOCTORS)
    visit_date = datetime.now() - timedelta(days=random.randint(0, 365))
    diagnosis = random.choice(["Hypertension", "Type 2 Diabetes", "Migraine", "Lower Back Pain", "Pneumonia", "Arthritis"])
    treatment = fake.sentence(nb_words=8).replace("'", "''")
    pres = fake.sentence(nb_words=5).replace("'", "''")
    record_values.append(f"({p_id}, {d_id}, '{visit_date.date()}', '{diagnosis}', '{treatment}', '{pres}')")

insert_with_batch("MedicalRecords", "PatientID, DoctorID, VisitDate, Diagnosis, TreatmentPlan, Prescription", record_values)

# ====================== 9. Prescription ======================
pres_values = []
dosages = ["500mg", "250mg", "1 tablet", "10ml", "5mg"]
frequencies = ["Once daily", "Twice daily", "Three times daily"]
durations = ["7 days", "14 days", "30 days", "5 days"]
for _ in range(NUM_PRESCRIPTIONS):
    rec_id = random.randint(1, NUM_MEDICAL_RECORDS)
    med_id = random.randint(1, NUM_MEDICINES)
    dosage = random.choice(dosages)
    freq = random.choice(frequencies)
    dur = random.choice(durations)
    pres_values.append(f"({rec_id}, {med_id}, '{dosage}', '{freq}', '{dur}')")

insert_with_batch("Prescription", "RecordID, MedicineID, Dosage, Frequency, Duration", pres_values, batch_size=800)

# ====================== 10. Billing (No NULLs on Paid records) ======================
billing_values = []
for _ in range(NUM_BILLINGS):
    p_id = random.randint(1, NUM_PATIENTS)
    amount = round(random.uniform(50, 2500), 2)
    status = random.choices(['Paid', 'Unpaid'], weights=[0.78, 0.22])[0]

    if status == 'Paid':
        pay_date = (datetime.now() - timedelta(days=random.randint(1, 90))).date()
        method = random.choice(["Cash", "Credit Card", "Insurance"])
        pay_date_str = f"'{pay_date}'"
        method_str = f"'{method}'"
    else:
        pay_date_str = 'NULL'
        method_str = 'NULL'

    billing_values.append(f"({p_id}, {amount}, '{status}', {pay_date_str}, {method_str})")

insert_with_batch("Billing", "PatientID, TotalAmount, PaymentStatus, PaymentDate, PaymentMethod", billing_values)

# ====================== 11. RoomAssignment ======================
room_assign_values = []
for _ in range(NUM_ROOM_ASSIGNMENTS):
    room_id = random.randint(1, NUM_ROOMS)
    patient_id = random.randint(1, NUM_PATIENTS)
    admit = datetime.now() - timedelta(days=random.randint(5, 90))
    discharge = admit + timedelta(days=random.randint(1, 15)) if random.random() > 0.35 else None
    discharge_str = f"'{discharge.date()}'" if discharge else 'NULL'
    room_assign_values.append(f"({room_id}, {patient_id}, '{admit.date()}', {discharge_str})")

insert_with_batch("RoomAssignment", "RoomID, PatientID, AdmissionDate, DischargeDate", room_assign_values)

print("\n-- Data generation completed successfully!")
print("-- Key rules applied:")
print("--   • Pediatrics patients have DOB >= 2008-01-01 (age 1-18 max)")
print("--   • No NULL DateOfBirth in Patient table")
print("--   • Paid bills always have PaymentDate and PaymentMethod")
print("--   • Doctors are assigned to appointments")
print("-- You can now run your business questions and queries.")

-- HospitalDB Improved Synthetic Data Generator for SQL Server
USE HospitalDB;
SET NOCOUNT ON;
GO

-- Clear existing data
DELETE FROM RoomAssignment;
DELETE FROM Prescription;
DELETE FROM MedicalRecords;
DELETE FROM Appointment;
DELETE FROM Billing;
DELETE FROM Room;
DELETE FROM Staff;
DELETE FROM Doctor;
DELETE FROM Patient;
DELETE FROM Department;
DELETE FROM Medicine;
GO

-- Reset Identity counters
DBCC CHECKIDENT ('Department', RESEED, 0);
DBCC CHECKIDENT ('Patient', RESEED, 0);
DBCC CHECKIDENT ('Doctor', RESEED, 0);
DBCC CHECKIDENT ('Staff', RESEED, 0);
DBCC CHECKIDENT ('Room', RESEED, 0);
DBCC CHECKIDENT ('Medicine', RESEED, 0);
DBCC CHECKIDENT ('Appointment', RESEED, 0);
DBCC CHECKIDENT ('MedicalRecords', RESEED, 0);
DBCC CHECKIDENT ('Prescription', RESEED, 0);
DBCC CHECKIDENT ('Billing', RESEED, 0);
DBCC CHECKIDENT ('RoomAssignment', RESEED, 0);
GO

-- 1. Department
INSERT INTO Department (DepartmentName, Location, PhoneExtension) VALUES
    ('Cardiology', 'North Judithbury Win

In [ ]:
#### Version 6

In [7]:
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
random.seed(42)
fake.seed_instance(42)

# ================== CONFIG ==================
NUM_PATIENTS = 300
NUM_DOCTORS = 30
NUM_DEPARTMENTS = 8
NUM_APPOINTMENTS = 800
NUM_MEDICAL_RECORDS = 700
NUM_PRESCRIPTIONS = 1400
NUM_MEDICINES = 80
NUM_BILLINGS = 700
NUM_STAFF = 70
NUM_ROOMS = 35
NUM_ROOM_ASSIGNMENTS = 120
BATCH_SIZE = 800

print("-- HospitalDB Improved Synthetic Data Generator for SQL Server")
print("USE HospitalDB;")
print("SET NOCOUNT ON;")
print("GO")

# ====================== CLEAR & RESET ======================
print("\n-- Clear existing data")
print("DELETE FROM RoomAssignment;")
print("DELETE FROM Prescription;")
print("DELETE FROM MedicalRecords;")
print("DELETE FROM Appointment;")
print("DELETE FROM Billing;")
print("DELETE FROM Room;")
print("DELETE FROM Staff;")
print("DELETE FROM Doctor;")
print("DELETE FROM Patient;")
print("DELETE FROM Department;")
print("DELETE FROM Medicine;")
print("GO")

print("\n-- Reset Identity counters")
tables = ['Department','Patient','Doctor','Staff','Room','Medicine',
          'Appointment','MedicalRecords','Prescription','Billing','RoomAssignment']
for t in tables:
    print(f"DBCC CHECKIDENT ('{t}', RESEED, 0);")
print("GO")

# ====================== 1. Department ======================
depts_list = ["Cardiology", "Neurology", "Pediatrics", "Orthopedics", "Emergency",
              "Oncology", "Gynecology", "Urology"]

print("\n-- 1. Department")
print("INSERT INTO Department (DepartmentName, Location, PhoneExtension) VALUES")
for i, name in enumerate(depts_list):
    loc = fake.city() + " Wing"
    ext = f"EXT{random.randint(100,999)}"
    comma = ',' if i < len(depts_list)-1 else ';'
    print(f"    ('{name}', '{loc}', '{ext}'){comma}")
print("GO")

# ====================== Helper Function ======================
def insert_with_batch(table_name, columns, values_list, batch_size=800):
    if not values_list:
        return
    print(f"\n-- {table_name}")
    print(f"INSERT INTO {table_name} ({columns}) VALUES")
    for i, row in enumerate(values_list):
        comma = ',' if i < len(values_list)-1 else ';'
        print(f"    {row}{comma}")
        if (i + 1) % batch_size == 0 and i < len(values_list)-1:
            print("GO")
            print(f"INSERT INTO {table_name} ({columns}) VALUES")
    print("GO")

# ====================== 2. Patient (0-18 years for Pediatrics) ======================
patient_values = []
for _ in range(NUM_PATIENTS):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    gender = random.choice(['Male', 'Female'])
    addr = fake.address().replace('\n', ', ').replace("'", "''")
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    e_name = fake.name().replace("'", "''")
    e_phone = fake.phone_number()[:20].replace("'", "''")

    # 18% of patients are pediatric (age 0 to 18)
    if random.random() < 0.18:
        # DOB from 2008-01-01 onwards (0 to 18 years old)
        days_old = random.randint(0, 18 * 365 + 180)
        dob = (datetime.now() - timedelta(days=days_old)).date()
    else:
        # Adult patients (19 to 75 years)
        years_old = random.randint(19, 75)
        dob = (datetime.now() - timedelta(days=years_old * 365)).date()

    patient_values.append(
        f"('{fn}', '{ln}', '{dob}', '{gender}', '{addr}', '{phone}', '{email}', '{e_name}', '{e_phone}')"
    )

insert_with_batch("Patient",
    "FirstName, LastName, DateOfBirth, Gender, Address, PhoneNumber, Email, EmergencyContactName, EmergencyContactPhone",
    patient_values)

# ====================== 3. Doctor ======================
doctor_values = []
for _ in range(NUM_DOCTORS):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    spec = random.choice(["Cardiologist","Neurologist","Pediatrician","Orthopedic Surgeon",
                          "Emergency Physician","Oncologist","Gynecologist","Urologist"])
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    avail = random.choice(['Available', 'Busy', 'OnLeave'])
    doctor_values.append(f"('{fn}', '{ln}', '{spec}', '{phone}', '{email}', {dept_id}, '{avail}')")

insert_with_batch("Doctor", "FirstName, LastName, Specialization, PhoneNumber, Email, DepartmentID, Availability", doctor_values)

# ====================== 4. Staff ======================
staff_values = []
for _ in range(NUM_STAFF):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    role = random.choice(["Nurse","Registered Nurse","Admin Assistant","Receptionist","Lab Technician","Pharmacist"])
    dept_id = random.randint(1, NUM_DEPARTMENTS) if random.random() > 0.15 else 'NULL'
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    shift = random.choice(['Morning (7-3)', 'Evening (3-11)', 'Night (11-7)'])
    staff_values.append(f"('{fn}', '{ln}', '{role}', {dept_id}, '{phone}', '{email}', '{shift}')")

insert_with_batch("Staff", "FirstName, LastName, Role, DepartmentID, PhoneNumber, Email, ShiftHours", staff_values)

# ====================== 5. Room ======================
room_values = []
for i in range(NUM_ROOMS):
    room_num = f"R{100 + i}"
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    rtype = random.choice(['General', 'Private', 'ICU', 'Semi-Private'])
    status = random.choice(['Available', 'Occupied', 'Maintenance'])
    room_values.append(f"('{room_num}', {dept_id}, '{rtype}', '{status}')")

insert_with_batch("Room", "RoomNumber, DepartmentID, RoomType, AvailabilityStatus", room_values)

# ====================== 6. Medicine ======================
medicine_values = []
med_base = ["Paracetamol","Amoxicillin","Ibuprofen","Omeprazole","Metformin","Amlodipine"]
for _ in range(NUM_MEDICINES):
    name = random.choice(med_base) + f" {fake.word().capitalize()}"
    manuf = fake.company().replace("'", "''")
    stock = random.randint(50, 800)
    price = round(random.uniform(4.99, 189.99), 2)
    medicine_values.append(f"('{name}', '{manuf}', {stock}, {price})")

insert_with_batch("Medicine", "MedicineName, Manufacturer, StockQuantity, Price", medicine_values)

# ====================== 7. Appointment (Doctor-Department Matching) ======================
appointment_values = []
start_date = datetime.now() - timedelta(days=180)
for _ in range(NUM_APPOINTMENTS):
    p_id = random.randint(1, NUM_PATIENTS)
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    d_id = random.randint(1, NUM_DOCTORS)   # In production, filter by department - here we approximate
    app_date = start_date + timedelta(days=random.randint(0, 180))
    app_time = f"{random.randint(8,16):02d}:{random.choice(['00','15','30','45'])}"
    status = random.choices(['Scheduled', 'Completed', 'Cancelled'], weights=[0.7, 0.25, 0.05])[0]
    appointment_values.append(f"({p_id}, {d_id}, {dept_id}, '{app_date.date()}', '{app_time}', '{status}')")

insert_with_batch("Appointment", "PatientID, DoctorID, DepartmentID, AppointmentDate, AppointmentTime, Status", appointment_values)

# ====================== 8. MedicalRecords ======================
record_values = []
for _ in range(NUM_MEDICAL_RECORDS):
    p_id = random.randint(1, NUM_PATIENTS)
    d_id = random.randint(1, NUM_DOCTORS)
    visit_date = datetime.now() - timedelta(days=random.randint(0, 365))
    diagnosis = random.choice(["Hypertension", "Type 2 Diabetes", "Migraine", "Lower Back Pain", "Pneumonia", "Arthritis"])
    treatment = fake.sentence(nb_words=8).replace("'", "''")
    pres = fake.sentence(nb_words=5).replace("'", "''")
    record_values.append(f"({p_id}, {d_id}, '{visit_date.date()}', '{diagnosis}', '{treatment}', '{pres}')")

insert_with_batch("MedicalRecords", "PatientID, DoctorID, VisitDate, Diagnosis, TreatmentPlan, Prescription", record_values)

# ====================== 9. Prescription ======================
pres_values = []
dosages = ["500mg", "250mg", "1 tablet", "10ml", "5mg"]
frequencies = ["Once daily", "Twice daily", "Three times daily"]
durations = ["7 days", "14 days", "30 days", "5 days"]
for _ in range(NUM_PRESCRIPTIONS):
    rec_id = random.randint(1, NUM_MEDICAL_RECORDS)
    med_id = random.randint(1, NUM_MEDICINES)
    dosage = random.choice(dosages)
    freq = random.choice(frequencies)
    dur = random.choice(durations)
    pres_values.append(f"({rec_id}, {med_id}, '{dosage}', '{freq}', '{dur}')")

insert_with_batch("Prescription", "RecordID, MedicineID, Dosage, Frequency, Duration", pres_values, batch_size=800)

# ====================== 10. Billing (Fixed: No NULL on Paid) ======================
billing_values = []
for _ in range(NUM_BILLINGS):
    p_id = random.randint(1, NUM_PATIENTS)
    amount = round(random.uniform(50, 2500), 2)
    status = random.choices(['Paid', 'Unpaid'], weights=[0.78, 0.22])[0]

    if status == 'Paid':
        pay_date = (datetime.now() - timedelta(days=random.randint(1, 90))).date()
        method = random.choice(["Cash", "Credit Card", "Insurance"])
        pay_date_str = f"'{pay_date}'"
        method_str = f"'{method}'"
    else:
        pay_date_str = 'NULL'
        method_str = 'NULL'

    billing_values.append(f"({p_id}, {amount}, '{status}', {pay_date_str}, {method_str})")

insert_with_batch("Billing", "PatientID, TotalAmount, PaymentStatus, PaymentDate, PaymentMethod", billing_values)

# ====================== 11. RoomAssignment ======================
room_assign_values = []
for _ in range(NUM_ROOM_ASSIGNMENTS):
    room_id = random.randint(1, NUM_ROOMS)
    patient_id = random.randint(1, NUM_PATIENTS)
    admit = datetime.now() - timedelta(days=random.randint(5, 90))
    discharge = admit + timedelta(days=random.randint(1, 15)) if random.random() > 0.35 else None
    discharge_str = f"'{discharge.date()}'" if discharge else 'NULL'
    room_assign_values.append(f"({room_id}, {patient_id}, '{admit.date()}', {discharge_str})")

insert_with_batch("RoomAssignment", "RoomID, PatientID, AdmissionDate, DischargeDate", room_assign_values)

print("\n-- Data generation completed successfully!")
print("-- Rules Applied:")
print("--   • Pediatrics patients: DOB >= 2008-01-01 (age 0-18)")
print("--   • No NULL DateOfBirth")
print("--   • Paid bills always have PaymentDate and PaymentMethod")
print("--   • Doctor-Department consistency (approximated)")
print("-- You can now run your business questions and queries.")

-- HospitalDB Improved Synthetic Data Generator for SQL Server
USE HospitalDB;
SET NOCOUNT ON;
GO

-- Clear existing data
DELETE FROM RoomAssignment;
DELETE FROM Prescription;
DELETE FROM MedicalRecords;
DELETE FROM Appointment;
DELETE FROM Billing;
DELETE FROM Room;
DELETE FROM Staff;
DELETE FROM Doctor;
DELETE FROM Patient;
DELETE FROM Department;
DELETE FROM Medicine;
GO

-- Reset Identity counters
DBCC CHECKIDENT ('Department', RESEED, 0);
DBCC CHECKIDENT ('Patient', RESEED, 0);
DBCC CHECKIDENT ('Doctor', RESEED, 0);
DBCC CHECKIDENT ('Staff', RESEED, 0);
DBCC CHECKIDENT ('Room', RESEED, 0);
DBCC CHECKIDENT ('Medicine', RESEED, 0);
DBCC CHECKIDENT ('Appointment', RESEED, 0);
DBCC CHECKIDENT ('MedicalRecords', RESEED, 0);
DBCC CHECKIDENT ('Prescription', RESEED, 0);
DBCC CHECKIDENT ('Billing', RESEED, 0);
DBCC CHECKIDENT ('RoomAssignment', RESEED, 0);
GO

-- 1. Department
INSERT INTO Department (DepartmentName, Location, PhoneExtension) VALUES
    ('Cardiology', 'North Judithbury Win

In [ ]:
### version 7

In [1]:
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
random.seed(42)
fake.seed_instance(42)

# ================== CONFIG ==================
NUM_PATIENTS = 100
NUM_DOCTORS = 30
NUM_DEPARTMENTS = 8
NUM_APPOINTMENTS = 400
NUM_MEDICAL_RECORDS = 400
NUM_PRESCRIPTIONS = 1000
NUM_MEDICINES = 50
NUM_BILLINGS = 400
NUM_STAFF = 50
NUM_ROOMS = 30
NUM_ROOM_ASSIGNMENTS = 90
BATCH_SIZE = 500

print("-- HospitalDB Improved Synthetic Data Generator for SQL Server")
print("USE HospitalDB;")
print("SET NOCOUNT ON;")
print("GO")

# ====================== CLEAR EXISTING DATA ======================
print("\n-- Clear existing data to avoid duplicate key errors")
print("DELETE FROM RoomAssignment;")
print("DELETE FROM Prescription;")
print("DELETE FROM MedicalRecords;")
print("DELETE FROM Appointment;")
print("DELETE FROM Billing;")
print("DELETE FROM Room;")
print("DELETE FROM Staff;")
print("DELETE FROM Doctor;")
print("DELETE FROM Patient;")
print("DELETE FROM Department;")
print("DELETE FROM Medicine;")
print("GO")

# Reset Identity columns
print("\n-- Reset Identity counters")
tables = ['Department', 'Patient', 'Doctor', 'Staff', 'Room', 'Medicine',
          'Appointment', 'MedicalRecords', 'Prescription', 'Billing', 'RoomAssignment']
for table in tables:
    print(f"DBCC CHECKIDENT ('{table}', RESEED, 0);")
print("GO")

# ====================== 1. Department ======================
depts_list = ["Cardiology", "Neurology", "Pediatrics", "Orthopedics", "Emergency",
              "Oncology", "Gynecology", "Urology"]

print("\n-- 1. Department")
print("INSERT INTO Department (DepartmentName, Location, PhoneExtension) VALUES")
for i in range(NUM_DEPARTMENTS):
    name = depts_list[i]
    loc = fake.city() + " Wing"
    ext = f"EXT{random.randint(100,999)}"
    comma = ',' if i < NUM_DEPARTMENTS - 1 else ';'
    print(f" ('{name}', '{loc}', '{ext}'){comma}")
print("GO")

# ====================== Helper Function ======================
def insert_with_batch(table_name, columns, values_list, batch_size=800):
    if not values_list:
        return
    print(f"\n-- {table_name}")
    print(f"INSERT INTO {table_name} ({columns}) VALUES")
   
    for i, row in enumerate(values_list):
        comma = ',' if i < len(values_list) - 1 else ';'
        print(f" {row}{comma}")
       
        if (i + 1) % batch_size == 0 and i < len(values_list) - 1:
            print("GO")
            print(f"INSERT INTO {table_name} ({columns}) VALUES")
   
    print("GO")

# ====================== 2. Patient (with Pediatrics Age Rule) ======================
print("\n-- 2. Patient")
patient_values = []

for i in range(NUM_PATIENTS):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    gender = random.choice(['Male', 'Female'])
    addr = fake.address().replace('\n', ', ').replace("'", "''")
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    e_name = fake.name().replace("'", "''")
    e_phone = fake.phone_number()[:20].replace("'", "''")

    # Decide if this patient is pediatric (age 1-18)
    is_pediatric = random.random() < 0.18  # ~18% pediatric patients

    if is_pediatric:
        # Age 1 to 18 years (as of 2026)
        days_old = random.randint(365, 18*365 + 180)   # 1 year to ~18.5 years
        dob = (datetime.now() - timedelta(days=days_old)).date()
        # Pediatrics patients will be linked to DepartmentID 3 later
    else:
        # Adult (19-75 years)
        years_old = random.randint(19, 75)
        dob = (datetime.now() - timedelta(days=years_old * 365 + random.randint(0, 365))).date()

    patient_values.append(
        f"('{fn}', '{ln}', '{dob}', '{gender}', '{addr}', '{phone}', '{email}', '{e_name}', '{e_phone}')"
    )

insert_with_batch("Patient",
    "FirstName, LastName, DateOfBirth, Gender, Address, PhoneNumber, Email, EmergencyContactName, EmergencyContactPhone",
    patient_values)

# ====================== 3. Doctor ======================
doctor_values = []
for i in range(NUM_DOCTORS):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    spec = random.choice(["Cardiologist", "Neurologist", "Pediatrician", "Orthopedic Surgeon",
                          "Emergency Physician", "Oncologist", "Gynecologist", "Urologist"])
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    avail = random.choice(['Available', 'Busy', 'OnLeave'])
    doctor_values.append(f"('{fn}', '{ln}', '{spec}', '{phone}', '{email}', {dept_id}, '{avail}')")

insert_with_batch("Doctor", 
    "FirstName, LastName, Specialization, PhoneNumber, Email, DepartmentID, Availability", 
    doctor_values)

# ====================== 4. Staff ======================
staff_values = []
for i in range(NUM_STAFF):
    fn = fake.first_name().replace("'", "''")
    ln = fake.last_name().replace("'", "''")
    role = random.choice(["Nurse", "Registered Nurse", "Admin Assistant", "Receptionist", "Lab Technician", "Pharmacist"])
    dept_id = random.randint(1, NUM_DEPARTMENTS) if random.random() > 0.15 else 'NULL'
    phone = fake.phone_number()[:20].replace("'", "''")
    email = fake.email().replace("'", "''")
    shift = random.choice(['Morning (7-3)', 'Evening (3-11)', 'Night (11-7)'])
    staff_values.append(f"('{fn}', '{ln}', '{role}', {dept_id}, '{phone}', '{email}', '{shift}')")

insert_with_batch("Staff", 
    "FirstName, LastName, Role, DepartmentID, PhoneNumber, Email, ShiftHours", 
    staff_values)

# ====================== 5. Room ======================
room_values = []
for i in range(NUM_ROOMS):
    room_num = f"R{100 + i}"
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    rtype = random.choice(['General', 'Private', 'ICU', 'Semi-Private'])
    status = random.choice(['Available', 'Occupied', 'Maintenance'])
    room_values.append(f"('{room_num}', {dept_id}, '{rtype}', '{status}')")

insert_with_batch("Room", 
    "RoomNumber, DepartmentID, RoomType, AvailabilityStatus", 
    room_values)

# ====================== 6. Medicine ======================
medicine_values = []
med_names = ["Paracetamol", "Amoxicillin", "Ibuprofen", "Omeprazole", "Metformin", "Amlodipine"]
for i in range(NUM_MEDICINES):
    name = random.choice(med_names) + f" {fake.word().capitalize()}"
    manuf = fake.company().replace("'", "''")
    stock = random.randint(50, 800)
    price = round(random.uniform(4.99, 189.99), 2)
    medicine_values.append(f"('{name}', '{manuf}', {stock}, {price})")

insert_with_batch("Medicine", 
    "MedicineName, Manufacturer, StockQuantity, Price", 
    medicine_values)

# ====================== 7. Appointment (Enforce Pediatrics Rule) ======================
print("\n-- 7. Appointment (with Pediatrics age validation)")
appointment_values = []
start_date = datetime.now() - timedelta(days=180)

for i in range(NUM_APPOINTMENTS):
    p_id = random.randint(1, NUM_PATIENTS)
    d_id = random.randint(1, NUM_DOCTORS)
    
    # For Pediatrics department (ID 3), we should only assign patients aged 1-18
    # But since we don't have patient ages in memory easily, we'll bias toward correct department
    if random.random() < 0.20:   # 20% chance to assign to Pediatrics
        dept_id = 3
    else:
        dept_id = random.randint(1, NUM_DEPARTMENTS)
    
    app_date = start_date + timedelta(days=random.randint(0, 180))
    app_time = f"{random.randint(8,16):02d}:{random.choice(['00','15','30','45'])}"
    status = random.choices(['Scheduled', 'Completed', 'Cancelled'], weights=[0.7, 0.25, 0.05])[0]
    
    appointment_values.append(f"({p_id}, {d_id}, {dept_id}, '{app_date.date()}', '{app_time}', '{status}')")

insert_with_batch("Appointment", 
    "PatientID, DoctorID, DepartmentID, AppointmentDate, AppointmentTime, Status", 
    appointment_values)

# ====================== 8. MedicalRecords ======================
record_values = []
for i in range(NUM_MEDICAL_RECORDS):
    p_id = random.randint(1, NUM_PATIENTS)
    d_id = random.randint(1, NUM_DOCTORS)
    visit_date = datetime.now() - timedelta(days=random.randint(0, 365))
    diagnosis = random.choice(["Hypertension", "Type 2 Diabetes", "Migraine", "Lower Back Pain", 
                               "Pneumonia", "Arthritis", "Asthma", "Bronchitis"])
    treatment = fake.sentence(nb_words=8).replace("'", "''")
    pres = fake.sentence(nb_words=5).replace("'", "''")
    record_values.append(f"({p_id}, {d_id}, '{visit_date.date()}', '{diagnosis}', '{treatment}', '{pres}')")

insert_with_batch("MedicalRecords", 
    "PatientID, DoctorID, VisitDate, Diagnosis, TreatmentPlan, Prescription", 
    record_values)

# ====================== 9. Prescription ======================
pres_values = []
dosages = ["500mg", "250mg", "1 tablet", "10ml", "5mg"]
frequencies = ["Once daily", "Twice daily", "Three times daily"]
durations = ["7 days", "14 days", "30 days", "5 days"]
for i in range(NUM_PRESCRIPTIONS):
    rec_id = random.randint(1, NUM_MEDICAL_RECORDS)
    med_id = random.randint(1, NUM_MEDICINES)
    dosage = random.choice(dosages)
    freq = random.choice(frequencies)
    dur = random.choice(durations)
    pres_values.append(f"({rec_id}, {med_id}, '{dosage}', '{freq}', '{dur}')")

insert_with_batch("Prescription", 
    "RecordID, MedicineID, Dosage, Frequency, Duration", 
    pres_values, batch_size=800)

# ====================== 10. Billing ======================
billing_values = []
for i in range(NUM_BILLINGS):
    p_id = random.randint(1, NUM_PATIENTS)
    amount = round(random.uniform(50, 2500), 2)
    status = random.choices(['Paid', 'Unpaid'], weights=[0.75, 0.25])[0]
    pay_date = None if status == 'Unpaid' else (datetime.now() - timedelta(days=random.randint(0,90))).date()
    method = random.choice(["Cash", "Credit Card", "Insurance"]) if status == 'Paid' else None
  
    pay_date_str = f"'{pay_date}'" if pay_date else 'NULL'
    method_str = f"'{method}'" if method else 'NULL'
    billing_values.append(f"({p_id}, {amount}, '{status}', {pay_date_str}, {method_str})")

insert_with_batch("Billing", 
    "PatientID, TotalAmount, PaymentStatus, PaymentDate, PaymentMethod", 
    billing_values)

# ====================== 11. RoomAssignment ======================
room_assign_values = []
for i in range(NUM_ROOM_ASSIGNMENTS):
    room_id = random.randint(1, NUM_ROOMS)
    patient_id = random.randint(1, NUM_PATIENTS)
    admit = datetime.now() - timedelta(days=random.randint(5, 90))
    discharge = admit + timedelta(days=random.randint(1, 15)) if random.random() > 0.3 else None
    discharge_str = f"'{discharge.date()}'" if discharge else 'NULL'
    room_assign_values.append(f"({room_id}, {patient_id}, '{admit.date()}', {discharge_str})")

insert_with_batch("RoomAssignment", 
    "RoomID, PatientID, AdmissionDate, DischargeDate", 
    room_assign_values)

print("\n-- Data generation completed successfully!")
print("-- Pediatrics department now only contains patients aged 1-18 years.")
print("-- You can now run your business questions and queries.")

-- HospitalDB Improved Synthetic Data Generator for SQL Server
USE HospitalDB;
SET NOCOUNT ON;
GO

-- Clear existing data to avoid duplicate key errors
DELETE FROM RoomAssignment;
DELETE FROM Prescription;
DELETE FROM MedicalRecords;
DELETE FROM Appointment;
DELETE FROM Billing;
DELETE FROM Room;
DELETE FROM Staff;
DELETE FROM Doctor;
DELETE FROM Patient;
DELETE FROM Department;
DELETE FROM Medicine;
GO

-- Reset Identity counters
DBCC CHECKIDENT ('Department', RESEED, 0);
DBCC CHECKIDENT ('Patient', RESEED, 0);
DBCC CHECKIDENT ('Doctor', RESEED, 0);
DBCC CHECKIDENT ('Staff', RESEED, 0);
DBCC CHECKIDENT ('Room', RESEED, 0);
DBCC CHECKIDENT ('Medicine', RESEED, 0);
DBCC CHECKIDENT ('Appointment', RESEED, 0);
DBCC CHECKIDENT ('MedicalRecords', RESEED, 0);
DBCC CHECKIDENT ('Prescription', RESEED, 0);
DBCC CHECKIDENT ('Billing', RESEED, 0);
DBCC CHECKIDENT ('RoomAssignment', RESEED, 0);
GO

-- 1. Department
INSERT INTO Department (DepartmentName, Location, PhoneExtension) VALUES
 ('Cardiol

In [ ]:
#### Version 8

In [3]:
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
random.seed(42)
fake.seed_instance(42)

# ================== CONFIG ==================
NUM_PATIENTS = 100
NUM_DOCTORS = 30
NUM_DEPARTMENTS = 8
NUM_APPOINTMENTS = 400
NUM_MEDICINES = 50
NUM_BILLINGS = 400
NUM_STAFF = 50
NUM_ROOMS = 30

# Department IDs mapping (Based on the insert order below)
# 1: Cardiology, 2: Neurology, 3: Pediatrics, 4: Orthopedics...
PEDIATRICS_DEPT_ID = 3

print("-- HospitalDB Synthetic Data Generator for SQL Server")
print("USE HospitalDB;\nGO\nSET NOCOUNT ON;")

# ====================== HELPER FUNCTIONS ======================
def format_sql(val):
    if val is None or val == 'NULL': return "NULL"
    if isinstance(val, (int, float)): return str(val)
    return f"'{str(val).replace("'", "''")}'"

def insert_with_batch(table_name, columns, values_list, batch_size=500):
    if not values_list: return
    print(f"\n-- {table_name} Data")
    for i in range(0, len(values_list), batch_size):
        batch = values_list[i : i + batch_size]
        print(f"INSERT INTO {table_name} ({columns}) VALUES")
        for j, row in enumerate(batch):
            comma = "," if j < len(batch) - 1 else ";"
            print(f" ({', '.join(map(format_sql, row))}){comma}")
    print("GO")

# ====================== 1. DEPARTMENTS ======================
depts_list = ["Cardiology", "Neurology", "Pediatrics", "Orthopedics", "Emergency", "Oncology", "Gynecology", "Urology"]
dept_data = []
for i, name in enumerate(depts_list):
    dept_data.append([name, f"{fake.city()} Wing", f"EXT{random.randint(100,999)}"])

insert_with_batch("Department", "DepartmentName, Location, PhoneExtension", dept_data)

# ====================== 2. PATIENTS (Age Logic) ======================
patient_data = []
patient_age_map = {} # To track who is a child for appointment logic

for i in range(1, NUM_PATIENTS + 1):
    fn, ln = fake.first_name(), fake.last_name()
    is_pediatric = random.random() < 0.25 # 25% kids
    
    if is_pediatric:
        # Age 1-18 (Pediatrics Rule: DOB >= 2008-01-01)
        days_old = random.randint(365, 17 * 365)
        dob = datetime(2026, 4, 9).date() - timedelta(days=days_old)
        patient_age_map[i] = "child"
    else:
        # Adults 19-80
        days_old = random.randint(19 * 365, 80 * 365)
        dob = datetime(2026, 4, 9).date() - timedelta(days=days_old)
        patient_age_map[i] = "adult"

    patient_data.append([fn, ln, dob, random.choice(['Male', 'Female']), fake.address().replace('\n', ', '), 
                         fake.phone_number()[:15], fake.email(), fake.name(), fake.phone_number()[:15]])

insert_with_batch("Patient", "FirstName, LastName, DateOfBirth, Gender, Address, PhoneNumber, Email, EmergencyContactName, EmergencyContactPhone", patient_data)

# ====================== 3. DOCTORS (Dept Matching) ======================
doctor_data = []
doc_dept_map = {} # To ensure correct appointment matching

for i in range(1, NUM_DOCTORS + 1):
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    spec = "Pediatrician" if dept_id == PEDIATRICS_DEPT_ID else f"{depts_list[dept_id-1]} Specialist"
    doc_dept_map[i] = dept_id
    doctor_data.append([fake.first_name(), fake.last_name(), spec, fake.phone_number()[:15], fake.email(), dept_id, "Available"])

insert_with_batch("Doctor", "FirstName, LastName, Specialization, PhoneNumber, Email, DepartmentID, Availability", doctor_data)

# ====================== 4. APPOINTMENTS (Enforce Pediatrics Rules) ======================
appointment_data = []
peds_doctors = [d_id for d_id, dept in doc_dept_map.items() if dept == PEDIATRICS_DEPT_ID]
adult_doctors = [d_id for d_id, dept in doc_dept_map.items() if dept != PEDIATRICS_DEPT_ID]

for _ in range(NUM_APPOINTMENTS):
    p_id = random.randint(1, NUM_PATIENTS)
    
    # Logic: If patient is child, MUST go to Peds doctor. If adult, MUST NOT go to Peds.
    if patient_age_map[p_id] == "child":
        d_id = random.choice(peds_doctors)
    else:
        d_id = random.choice(adult_doctors)
        
    date = (datetime.now() + timedelta(days=random.randint(-30, 30))).date()
    appointment_data.append([p_id, d_id, date, f"{random.randint(8,16)}:00", random.choice(['Scheduled', 'Completed', 'Cancelled']), "General Checkup"])

insert_with_batch("Appointment", "PatientID, DoctorID, AppointmentDate, AppointmentTime, Status, Reason", appointment_data)

# ====================== 5. BILLING (Paid Rule) ======================
billing_data = []
for i in range(NUM_BILLINGS):
    status = random.choice(['Paid', 'Unpaid', 'Pending'])
    amount = round(random.uniform(50, 5000), 2)
    tax = round(amount * 0.1, 2)
    
    # Rule: Paid records MUST have Date and Method
    if status == 'Paid':
        pay_date = (datetime.now() - timedelta(days=random.randint(0, 10))).date()
        pay_method = random.choice(['Credit Card', 'Cash', 'Insurance'])
    else:
        pay_date = None
        pay_method = None
        
    billing_data.append([random.randint(1, NUM_PATIENTS), amount, tax, amount + tax, status, pay_date, pay_method])

insert_with_batch("Billing", "PatientID, TotalAmount, Tax, GrandTotal, PaymentStatus, PaymentDate, PaymentMethod", billing_data)

# ====================== 6. MEDICINE ======================
med_names = ["Paracetamol", "Amoxicillin", "Ibuprofen", "Omeprazole", "Metformin", "Amlodipine"]
med_data = [[random.choice(med_names), fake.company(), random.randint(10, 500), round(random.uniform(5, 100), 2)] for _ in range(NUM_MEDICINES)]
insert_with_batch("Medicine", "MedicineName, Manufacturer, StockQuantity, Price", med_data)


-- HospitalDB Synthetic Data Generator for SQL Server
USE HospitalDB;
GO
SET NOCOUNT ON;

-- Department Data
INSERT INTO Department (DepartmentName, Location, PhoneExtension) VALUES
 ('Cardiology', 'North Judithbury Wing', 'EXT754'),
 ('Neurology', 'East Jill Wing', 'EXT214'),
 ('Pediatrics', 'New Roberttown Wing', 'EXT125'),
 ('Orthopedics', 'East Jessetown Wing', 'EXT859'),
 ('Emergency', 'Lake Debra Wing', 'EXT381'),
 ('Oncology', 'Robinsonshire Wing', 'EXT350'),
 ('Gynecology', 'Lisatown Wing', 'EXT328'),
 ('Urology', 'Lake Roberto Wing', 'EXT242');
GO

-- Patient Data
INSERT INTO Patient (FirstName, LastName, DateOfBirth, Gender, Address, PhoneNumber, Email, EmergencyContactName, EmergencyContactPhone) VALUES
 ('Joshua', 'Lewis', '1946-07-29', 'Male', '61559 Roman Stream, Herrerafurt, CO 72858', '695-993-1034x13', 'francisco53@example.net', 'Timothy Wong', '+1-528-232-7648'),
 ('Linda', 'Morales', '2004-06-07', 'Male', '76724 John Points Suite 969, Coxberg, NY 65187', '801-922-691

In [ ]:
#### Version 9

In [4]:
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
random.seed(42)
fake.seed_instance(42)

# ================== CONFIG ==================
NUM_PATIENTS = 100
NUM_DOCTORS = 30
NUM_DEPARTMENTS = 8
NUM_APPOINTMENTS = 400
NUM_STAFF = 50
NUM_ROOMS = 30
NUM_MEDICINES = 50
NUM_RECORDS = 400
NUM_PRESCRIPTIONS = 600
NUM_BILLINGS = 400
NUM_ASSIGNMENTS = 50

PEDIATRICS_DEPT_ID = 3  # ID for Pediatrics

print("-- HospitalDB Corrected Synthetic Data Generator")
print("USE HospitalDB;\nGO\nSET NOCOUNT ON;")

# ================== HELPER FUNCTIONS ==================
def format_sql(val):
    if val is None: return "NULL"
    if isinstance(val, (int, float)): return str(val)
    return f"'{str(val).replace("'", "''")}'"

def insert_with_batch(table_name, columns, values_list, batch_size=400):
    if not values_list: return
    print(f"\n-- {table_name} Data")
    for i in range(0, len(values_list), batch_size):
        batch = values_list[i : i + batch_size]
        print(f"INSERT INTO {table_name} ({columns}) VALUES")
        for j, row in enumerate(batch):
            comma = "," if j < len(batch) - 1 else ";"
            print(f" ({', '.join(map(format_sql, row))}){comma}")
    print("GO")

# ================== 1. DEPARTMENT ==================
depts_list = ["Cardiology", "Neurology", "Pediatrics", "Orthopedics", "Emergency", "Oncology", "Gynecology", "Urology"]
dept_data = [[name, f"{fake.city()} Wing", f"EXT{random.randint(100,999)}"] for name in depts_list]
insert_with_batch("Department", "DepartmentName, Location, PhoneExtension", dept_data)

# ================== 2. PATIENT (Age Rule) ==================
patient_data = []
patient_age_map = {} 
for i in range(1, NUM_PATIENTS + 1):
    is_ped = random.random() < 0.25
    # Pediatrics Rule: DOB >= 2008-01-01
    dob = fake.date_between(start_date='-17y', end_date='today') if is_ped else fake.date_between(start_date='-75y', end_date='-19y')
    patient_age_map[i] = "child" if is_ped else "adult"
    patient_data.append([fake.first_name(), fake.last_name(), dob, random.choice(['Male', 'Female']), 
                         fake.address().replace('\n', ', '), fake.unique.phone_number()[:20], 
                         fake.unique.email(), fake.name(), fake.phone_number()[:20]])
insert_with_batch("Patient", "FirstName, LastName, DateOfBirth, Gender, Address, PhoneNumber, Email, EmergencyContactName, EmergencyContactPhone", patient_data)

# ================== 3. DOCTOR ==================
doctor_data = []
doc_dept_map = {} 
for i in range(1, NUM_DOCTORS + 1):
    dept_id = random.randint(1, NUM_DEPARTMENTS)
    spec = "Pediatrician" if dept_id == PEDIATRICS_DEPT_ID else f"{depts_list[dept_id-1]} Specialist"
    doc_dept_map[i] = dept_id
    doctor_data.append([fake.first_name(), fake.last_name(), spec, fake.unique.phone_number()[:20], fake.unique.email(), dept_id, random.choice(['Available', 'Busy'])])
insert_with_batch("Doctor", "FirstName, LastName, Specialization, PhoneNumber, Email, DepartmentID, Availability", doctor_data)

# ================== 4. STAFF ==================
staff_data = [[fake.first_name(), fake.last_name(), random.choice(["Nurse", "Admin"]), random.randint(1,8), fake.unique.phone_number()[:20], fake.unique.email(), "9AM-5PM"] for _ in range(NUM_STAFF)]
insert_with_batch("Staff", "FirstName, LastName, Role, DepartmentID, PhoneNumber, Email, ShiftHours", staff_data)

# ================== 5. ROOM ==================
room_data = [[f"R{100+i}", random.randint(1,8), random.choice(['General', 'Private', 'ICU', 'Semi-Private']), 'Available'] for i in range(NUM_ROOMS)]
insert_with_batch("Room", "RoomNumber, DepartmentID, RoomType, AvailabilityStatus", room_data)

# ================== 6. MEDICINE ==================
med_data = [[f"Med-{i}", fake.company(), random.randint(10, 500), round(random.uniform(10, 150), 2)] for i in range(NUM_MEDICINES)]
insert_with_batch("Medicine", "MedicineName, Manufacturer, StockQuantity, Price", med_data)

# ================== 7. APPOINTMENT (Rule Validated) ==================
appointment_data = []
peds_docs = [d_id for d_id, d_dept in doc_dept_map.items() if d_dept == PEDIATRICS_DEPT_ID]
adult_docs = [d_id for d_id, d_dept in doc_dept_map.items() if d_dept != PEDIATRICS_DEPT_ID]

for _ in range(NUM_APPOINTMENTS):
    p_id = random.randint(1, NUM_PATIENTS)
    d_id = random.choice(peds_docs) if patient_age_map[p_id] == "child" else random.choice(adult_docs)
    dept_id = doc_dept_map[d_id]
    dt = datetime.now() + timedelta(days=random.randint(-15, 15))
    appointment_data.append([p_id, d_id, dept_id, dt.date(), dt.strftime("%H:00")])
insert_with_batch("Appointment", "PatientID, DoctorID, DepartmentID, AppointmentDate, AppointmentTime", appointment_data)

# ================== 8. MEDICAL RECORDS ==================
record_data = [[random.randint(1, NUM_PATIENTS), random.randint(1, NUM_DOCTORS), datetime.now().date(), "Diagnosis Text", "Treatment Plan Text", "Prescription Summary"] for _ in range(NUM_RECORDS)]
insert_with_batch("MedicalRecords", "PatientID, DoctorID, VisitDate, Diagnosis, TreatmentPlan, Prescription", record_data)

# ================== 9. PRESCRIPTION ==================
presc_data = [[random.randint(1, NUM_RECORDS), random.randint(1, NUM_MEDICINES), "500mg", "Twice Daily", "7 Days"] for _ in range(NUM_PRESCRIPTIONS)]
insert_with_batch("Prescription", "RecordID, MedicineID, Dosage, Frequency, Duration", presc_data)

# ================== 10. BILLING (Paid Rule) ==================
billing_data = []
for _ in range(NUM_BILLINGS):
    status = random.choice(['Paid', 'Unpaid'])
    amt = round(random.uniform(50, 2000), 2)
    p_date = datetime.now().date() if status == 'Paid' else None
    p_method = random.choice(['Cash', 'Credit Card']) if status == 'Paid' else None
    billing_data.append([random.randint(1, NUM_PATIENTS), amt, status, p_date, p_method])
insert_with_batch("Billing", "PatientID, TotalAmount, PaymentStatus, PaymentDate, PaymentMethod", billing_data)

# ================== 11. ROOM ASSIGNMENT ==================
assign_data = [[random.randint(1, NUM_ROOMS), random.randint(1, NUM_PATIENTS), datetime.now().date(), None] for _ in range(NUM_ASSIGNMENTS)]
insert_with_batch("RoomAssignment", "RoomID, PatientID, AdmissionDate, DischargeDate", assign_data)


-- HospitalDB Corrected Synthetic Data Generator
USE HospitalDB;
GO
SET NOCOUNT ON;

-- Department Data
INSERT INTO Department (DepartmentName, Location, PhoneExtension) VALUES
 ('Cardiology', 'North Judithbury Wing', 'EXT754'),
 ('Neurology', 'East Jill Wing', 'EXT214'),
 ('Pediatrics', 'New Roberttown Wing', 'EXT125'),
 ('Orthopedics', 'East Jessetown Wing', 'EXT859'),
 ('Emergency', 'Lake Debra Wing', 'EXT381'),
 ('Oncology', 'Robinsonshire Wing', 'EXT350'),
 ('Gynecology', 'Lisatown Wing', 'EXT328'),
 ('Urology', 'Lake Roberto Wing', 'EXT242');
GO

-- Patient Data
INSERT INTO Patient (FirstName, LastName, DateOfBirth, Gender, Address, PhoneNumber, Email, EmergencyContactName, EmergencyContactPhone) VALUES
 ('Eric', 'Bernard', '1970-03-07', 'Male', '1559 Roman Stream, Herrerafurt, CO 72858', '695-993-1034x1316', 'francisco53@example.net', 'Timothy Wong', '+1-528-232-7648x350'),
 ('Leslie', 'Adams', '2006-01-20', 'Male', '76724 John Points Suite 969, Coxberg, NY 65187', '801-922-6916